# Image Classification

In [1]:
!pip install -U datasets transformers torchvision accelerate


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [17]:
# pip install -U datasets transformers accelerate torchvision

import torch
import numpy as np
import torchvision.transforms as T
from datasets import load_dataset, Image
from transformers import (
    ViTImageProcessor,
    ViTForImageClassification,
    TrainingArguments,
    Trainer,
)

In [32]:
# pip install -U datasets transformers accelerate torchvision

import torch
import torchvision.transforms as T
from datasets import load_dataset, Image, DatasetDict
from transformers import ViTImageProcessor, ViTForImageClassification, TrainingArguments, Trainer

# 1) Load dataset and ensure PIL decoding
data_dir = "./image-dataset/bottle_cls"  # expects train/<class>/*.png and test/<class>/*.png
ds = load_dataset("imagefolder", data_dir=data_dir)  # returns DatasetDict with 'train' and 'test' splits
ds = ds.cast_column("image", Image(decode=True))  # decode to PIL so transforms can run [keeps PIL] 

# 2) Drop rows with missing labels (guards against None-typed labels)
def has_label(ex):
    return ex["label"] is not None

ds = DatasetDict({
    split: split_ds.filter(has_label, desc=f"Filter None labels [{split}]")
    for split, split_ds in ds.items()
})

# 3) Build label maps from the cleaned training split
labels = ds["train"].features["label"].names  # e.g., ['broken_large','broken_small','contamination','good']
id2label = {i: c for i, c in enumerate(labels)}
label2id = {c: i for i, c in enumerate(labels)}

# 4) Processor + torchvision transforms (use ImageNet stats)
checkpoint = "google/vit-base-patch16-224"
processor = ViTImageProcessor.from_pretrained(checkpoint)  # provides mean/std, size, etc.
normalize = T.Normalize(mean=processor.image_mean, std=processor.image_std)
tfm_train = T.Compose([T.Resize((224, 224)), T.RandomHorizontalFlip(), T.ToTensor(), normalize])
tfm_eval  = T.Compose([T.Resize((224, 224)), T.ToTensor(), normalize])

# 5) Offline preprocessing to pixel_values (Tensor) and labels (int)
def preprocess_batch(examples, tfm):
    pil_imgs = examples["image"]                        # list[PIL.Image]
    lab_vals = examples["label"]                        # list[int] or possibly numpy ints
    # Filter out any None that slipped through (belt-and-braces)
    keep = [i for i, lv in enumerate(lab_vals) if lv is not None]
    pil_imgs = [pil_imgs[i] for i in keep]
    lab_vals = [int(lab_vals[i]) for i in keep]
    tensors = [tfm(img.convert("RGB")) for img in pil_imgs]  # list[Tensor CxHxW]
    return {"pixel_values": tensors, "labels": lab_vals}     # key must be 'labels'

train_p = ds["train"].map(lambda ex: preprocess_batch(ex, tfm_train),
                          batched=True, remove_columns=["image", "label"])
test_p  = ds["test"].map(lambda ex: preprocess_batch(ex, tfm_eval),
                          batched=True, remove_columns=["image", "label"])

# 6) Format as torch so default collator can batch without custom code
train_p.set_format(type="torch", columns=["pixel_values", "labels"])
test_p.set_format(type="torch", columns=["pixel_values", "labels"])

# Optional sanity checks
ex = train_p[0]
assert isinstance(ex["pixel_values"], torch.Tensor) and isinstance(ex["labels"], torch.Tensor), "Expect tensors"
assert ex["pixel_values"].ndim == 3 and ex["pixel_values"].shape[-2:] == (224, 224), "Expect CxHxW tensors"

# 7) Initialize model (fresh classifier head for 4 classes)
model = ViTForImageClassification.from_pretrained(
    checkpoint,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,  # replaces the 1000-logit ImageNet head
)

# 8) TrainingArguments — keep labels and inputs by disabling pruning if needed
args = TrainingArguments(
    output_dir="vit-bottle-4class",
    eval_strategy="epoch",          # if this errors, use evaluation_strategy="epoch" per your install
    save_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    learning_rate=3e-5,
    weight_decay=0.05,
    logging_steps=50,
    load_best_model_at_end=True,
    remove_unused_columns=False,    # prevents Trainer from pruning 'labels' accidentally
)

# 9) Trainer — default collator works with set_format("torch")
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_p,
    eval_dataset=test_p,
    processing_class=processor,     # preferred over 'tokenizer' in newer versions
)

trainer.train()


Map: 100%|██████████| 83/83 [00:01<00:00, 59.33 examples/s]


ValueError: Columns ['pixel_values', 'labels'] not in the dataset. Current columns in the dataset: []

In [25]:
print(ds)
print(ds["train"].features)

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 209
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 83
    })
})
{'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['broken_large', 'broken_small', 'contamination', 'good'])}


In [26]:
ds.column_names

{'train': ['image', 'label'], 'test': ['image', 'label']}